# Pre filtered runs part of the AugComparison Sweep

All runs were in the same project, so simply running the following command in the `QueShell` will create the **runs.json** file:

```bash
list old -f wandb project -c x == AugComparison -s results test average_loss -op ~/Code/SLR/src/results/aug_comparison/runs.json
```

See the sweep [README.md](../../configfiles/AugComparison/README.md).

In [1]:
import json
from pathlib import Path

# locals
import pandas as pd

from src.que.core import CompExpInfo
from src.run_types import CUTOFF_9_NAMES, RESULTS_DIR


def load_runs(runs_path: Path) -> list[CompExpInfo]:
    with open(runs_path, "r") as f:
        return [CompExpInfo.model_validate(r) for r in json.load(f)]


In [2]:
results_dir = RESULTS_DIR / 'aug_comparison'
runs = load_runs(results_dir / 'runs.json')
print(f'{len(runs)} found')

Please update your PyTorchVideo to latest master
44 found


## Now we can compare the runs

In [3]:
avail_acc_types = ["top_k_average_per_class_acc", "top_k_per_instance_acc"]
acc_type = avail_acc_types[1]
set_names = ['test', 'val']

#### Convert to DataFrame

In [4]:
def get_crop_name(run: CompExpInfo) -> str:
    assert run.data.train_augs is not None, "Spatial augmentations are None"
    assert run.data.train_augs.spatial_aug is not None, "Spatial augmentations are None"
    return run.data.train_augs.spatial_aug[0].type

def get_sampler_name(run: CompExpInfo) -> str:
    assert run.data.train_augs is not None, "Temporal augmentations are None"
    assert run.data.train_augs.temporal_aug is not None, "Temporal augmentations are None"
    return run.data.train_augs.temporal_aug[0].type

keys = ["policy", "num_ops", "magnitude", "mean", "std","max_wobble", "speed_min", "speed_max",]

def get_parameters_and_values(run: CompExpInfo) -> dict:
    assert run.data.train_augs is not None, "Train augmentations are None"
    assert run.data.train_augs.spatial_aug is not None, "Spatial augmentations are None"
    assert run.data.train_augs.temporal_aug is not None, "Temporal augmentations are None"

    key_value_pairs = {}


    for aug in run.data.train_augs.spatial_aug + run.data.train_augs.temporal_aug:
        aug_dict = aug.model_dump()
        match_keys = [k for k in keys if k in aug_dict]
        values = [aug_dict[k] for k in match_keys]
        
        key_value_pairs.update({k: v for k, v in zip(match_keys, values)})
        
    return key_value_pairs
    

def unpack_parameters_and_values(run: CompExpInfo) -> dict:
    #max three parameters, fill columns with blank -
    params_and_values = get_parameters_and_values(run)
    params = []
    keys = list(params_and_values.keys())
    for i in range(3):
        params.append(f"{keys[i]}: {params_and_values[keys[i]]}" if i < len(keys) else "-")
    
    return {f"param_{i+1}": params[i] for i in range(3)}

df_format = [
    {
        "exp no": run.admin.exp_no,
        "run_id": run.wandb.run_id,
        "subset": run.admin.split,
        "type": "control" if len(run.wandb.tags) == 1 else run.wandb.tags[0],
        "strategy": run.wandb.tags[-1], 
        # "crop": get_crop_name(run),
        # "sampler": get_sampler_name(run),
    }
    | unpack_parameters_and_values(run)
    | {f'{set_name} {k}': v for set_name in set_names for k, v in run.results.model_dump()[set_name][acc_type].items()}
    | {
        "best_val_acc": run.results.best_val_acc,
        "best_val_loss": run.results.best_val_loss,
        "test_loss": run.results.test.average_loss,
        "config path": run.admin.config_path,
        }
    for run in runs
]

df = pd.DataFrame(df_format)

#Rename columns and format values
ns = [1, 5, 10]
for set_name in set_names:
    for n in ns:
        old_name, new_name = f"{set_name} top{n}", f"{set_name.capitalize()} Top-{n}"

        df = df.rename(columns={old_name: new_name})
        df[new_name] = df[new_name].apply(lambda x: f"{x * 100:.2f}")

In [5]:
subsets = CUTOFF_9_NAMES[:1]
# subsets = get_avail_splits()
for set_name in subsets:
    subdf = df[df['subset'] == set_name]
    print(f'{set_name}'.capitalize())
    # display(subdf.sort_values('Test Top-1', ascending=False))
    # display(subdf.sort_values('best_val_loss', ascending=True))
    display(subdf.sort_values('test_loss', ascending=True))

Asl100_cutoff_9


,exp no,run_id,subset,type,strategy,param_1,param_2,param_3,Test Top-1,Test Top-5,Test Top-10,Val Top-1,Val Top-5,Val Top-10,best_val_acc,best_val_loss,test_loss,config path
0,024,p6bqv0tv,asl100_cutoff_9,temporal,chunked,max_wobble: 0,-,-,76.74,93.02,96.12,77.51,92.01,97.93,79.881657,0.859231,1.005772,/home/luke/Code/SLR/src/configfiles/AugCompari...
1,030,nwqqpima,asl100_cutoff_9,temporal,wobble,max_wobble: 6,-,-,74.81,91.86,95.35,78.11,94.08,97.93,78.106509,0.832664,1.010051,/home/luke/Code/SLR/src/configfiles/AugCompari...
2,033,8blf7r20,asl100_cutoff_9,temporal,focal_norm,mean: 0.5,std: 0.35,max_wobble: 0,75.97,91.86,96.90,77.51,94.67,97.63,77.514793,0.808858,1.020266,/home/luke/Code/SLR/src/configfiles/AugCompari...
3,035,otgqfh3u,asl100_cutoff_9,spatial,RandAug,num_ops: 2,magnitude: 6,max_wobble: 0,74.42,91.09,95.35,74.56,93.49,97.63,75.147929,0.973231,1.021465,/home/luke/Code/SLR/src/configfiles/AugCompari...
4,037,jcht1af5,asl100_cutoff_9,spatial,RandAug,num_ops: 2,magnitude: 10,max_wobble: 0,74.81,93.02,95.35,77.22,93.49,96.45,77.218935,0.860436,1.023947,/home/luke/Code/SLR/src/configfiles/AugCompari...
5,035,p40wp9kx,asl100_cutoff_9,spatial,RandAug,num_ops: 2,magnitude: 6,max_wobble: 0,71.32,92.25,96.12,74.56,91.42,95.27,76.627219,0.964799,1.028317,/home/luke/Code/SLR/src/configfiles/AugCompari...
6,044,2sun0g6k,asl100_cutoff_9,spatial,AutoAug,max_wobble: 0,-,-,72.87,92.64,94.57,75.74,92.60,96.75,76.627219,0.929062,1.041553,/home/luke/Code/SLR/src/configfiles/AugCompari...
7,030,ijq0o5nu,asl100_cutoff_9,temporal,wobble,max_wobble: 6,-,-,74.81,92.64,95.74,79.29,95.27,98.52,79.289941,0.812260,1.054921,/home/luke/Code/SLR/src/configfiles/AugCompari...
8,037,z9ll39k2,asl100_cutoff_9,spatial,RandAug,num_ops: 2,magnitude: 10,max_wobble: 0,73.64,92.25,95.35,71.89,91.72,95.56,76.627219,1.051620,1.061296,/home/luke/Code/SLR/src/configfiles/AugCompari...
9,022,4kgmrq5i,asl100_cutoff_9,control,baseline,max_wobble: 0,-,-,69.77,93.02,96.12,74.85,94.08,97.04,76.331361,0.920498,1.074033,/home/luke/Code/SLR/src/configfiles/AugCompari...


In [8]:


# Column used to select the best run per model/subset
SELECTION_COL = "best_val_acc"   # change to "best_val_loss" if needed

# Optional: if you want custom model names with \cite, set escape=False below
# and store the LaTeX string directly in the 'model' column.
# ==========================================================

def make_latex_table_for_subset(df, subset, group_by = "model"):
    """Select best run per model and return LaTeX table for one subset."""
    sub = df[df["subset"] == subset].copy()
    sub[group_by] = sub[group_by].astype(str).str.strip()

    # Convert selection column to numeric and drop rows with missing value
    sub[SELECTION_COL] = pd.to_numeric(sub[SELECTION_COL], errors="coerce")
    sub = sub.dropna(subset=[SELECTION_COL])

    # Select best run: highest accuracy or lowest loss
    if SELECTION_COL == "best_val_loss":
        idx = sub.groupby(group_by)[SELECTION_COL].idxmin()
    else:
        idx = sub.groupby(group_by)[SELECTION_COL].idxmax()
    best = sub.loc[idx].copy()

    # Keep only needed columns and rename
    best = best[[group_by, "Test Top-1", "Test Top-5", "Test Top-10"]].rename(
        columns={
            group_by: "Model",
            "Test Top-1": "Acc@1",
            "Test Top-5": "Acc@5",
            "Test Top-10": "Acc@10",
        }
    )

    # Ensure metrics are numeric (to_latex will format them)
    for col in ["Acc@1", "Acc@5", "Acc@10"]:
        best[col] = pd.to_numeric(best[col], errors="coerce")

    # Generate LaTeX using .to_latex()
    latex = best.to_latex(
        index=False,
        na_rep="-",
        float_format="%.2f",
        caption=f"{subset} Results",
        label=f"tab:{subset.lower().replace('-', '_')}",
        position="htbp",
    )
    return latex


# Generate and print each table
for subset in subsets:
    print(f"% ===== Table for {subset} =====")
    print(make_latex_table_for_subset(df, subset, group_by="strategy"))
    print("\n")   # blank line between tables

% ===== Table for asl100_cutoff_9 =====
\begin{table}[htbp]
\caption{asl100_cutoff_9 Results}
\label{tab:asl100_cutoff_9}
\begin{tabular}{lrrr}
\toprule
Model & Acc@1 & Acc@5 & Acc@10 \\
\midrule
AutoAug & 72.48 & 91.86 & 96.12 \\
RandAug & 71.71 & 90.70 & 94.96 \\
RandomResizeCrop & 71.32 & 94.57 & 96.12 \\
baseline & 70.93 & 91.09 & 95.35 \\
chunked & 76.74 & 93.02 & 96.12 \\
focal_norm & 73.26 & 90.31 & 95.35 \\
no_aug & 66.67 & 90.31 & 94.96 \\
speed & 58.53 & 84.50 & 92.25 \\
wobble & 74.81 & 92.64 & 95.74 \\
\bottomrule
\end{tabular}
\end{table}



